# Huấn luyện Custom ResNet-18 Ultimate (`02e`, Train From Scratch, 45 Epochs)
Triển khai trọn bộ **2 Giai đoạn Cải thiện** (vẫn dùng lại nguyên file `dataset.zip` cũ, giữ nguyên `02a–02d` làm mốc đối chiếu):

### Giai đoạn 1 — Chuẩn bị & Sửa dữ liệu đầu vào (`Cell 2`)
1. **`PadToSquare` (Train + Val + Test):** Đệm viền bằng màu trung bình tập Train để ảnh chữ nhật `AAF (309×398)` không còn bị ép bẹt chiều dọc khi resize về `224×224`.
2. **`RandomDownscaleUpscale (48–112px)`:** Khử hiện tượng học đường tắt theo độ mờ của `AFAD-Full (40–61px)`.
3. **Domain Augmentation cho `FairFace` / `UTKFace`:** `RandomResizedCrop`, `RandomPerspective`, `GaussianBlur`, `RandomErasing (p=0.15)`.
4. **Lấy mẫu 2 Pha (chống `Prior Shift`):** Pha 1 (`Ep 1–32`) dùng **Half-Balanced (`1 / sqrt(n)`)**, Pha 2 (`Ep 33–45`) chuyển sang **phân bố tự nhiên (`shuffle=True`)**.

### Giai đoạn 2 — Kiến trúc Hybrid + Soft Ordinal LDL + EMA + Calibration (`Cell 3–9`)
1. **Kiến trúc `ResNet18HybridUltimate`:** `Deep Stem 3×3` + `ResNet-D` + `CBAM` + **`MHSA 4-heads (7×7)`** + `Multi-Scale (768-d)` + **Cross-Task Conditioning (`Gender 256-d -> Age Head`)**.
2. **Loss `coral_v4`:** **Soft Ordinal Labels (LDL)** dựa trên tuổi số thực của `AFAD-Full / AAF / UTKFace` + Weighted BCE + Phạt đơn điệu mềm + **Kendall Uncertainty Weighting** tự cân bằng Gender vs Age.
3. **Huấn luyện ổn định 45 Epochs:** Warm-up 5 epochs, `weight_decay = 5e-4`, `GradClip = 1.0`, **Model EMA (`decay = 0.995` + đồng bộ BN buffers)**, lưu song song `best_loss.pth` và **`best_acc.pth` (`0.4*Gender + 0.6*Age`)**.
4. **Suy luận chuẩn:** **Isotonic Monotonicity (`0.00%` vi phạm)** + **8-Threshold Calibration trên tập `Val`** + **TTA (`Orig + HFlip`)**.

In [ ]:
# [Cell 1] Thiết lập môi trường (Colab hỗ trợ dataset.zip / Local), GPU & Seed
import os, sys, random, json, time, shutil, copy
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

AGE_MODE = "coral_v4"
EXP_TAG = "resnet18_typeE_ultimate"
IN_COLAB = "google.colab" in sys.modules
DRIVE_SAVE_DIR = None
IMG_ROOT_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_DATASET_ZIP = "/content/drive/MyDrive/dataset.zip"
    DRIVE_PROJECT_DIR = "/content/drive/MyDrive/FINAL-PROJECT"
    DRIVE_SAVE_DIR = f"/content/drive/MyDrive/{EXP_TAG}_outputs"
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

    if os.path.exists(DRIVE_DATASET_ZIP):
        if not os.path.exists("/content/dataset/metadata/train.csv"):
            print(f"Đang giải nén {DRIVE_DATASET_ZIP} sang /content/ ...")
            !unzip -q "{DRIVE_DATASET_ZIP}" -d "/content"
        if os.path.exists("/content/dataset/dataset/metadata/train.csv"):
            os.chdir("/content/dataset")
        elif os.path.exists("/content/dataset/metadata/train.csv"):
            os.chdir("/content")
        else:
            raise FileNotFoundError("Không tìm thấy dataset/metadata/train.csv sau khi giải nén dataset.zip!")
    elif os.path.exists(DRIVE_PROJECT_DIR):
        os.chdir(DRIVE_PROJECT_DIR)
    else:
        raise FileNotFoundError("Không tìm thấy /content/drive/MyDrive/dataset.zip!")
else:
    cur = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(cur, "dataset", "metadata")) and cur != os.path.dirname(cur):
        cur = os.path.dirname(cur)
    os.chdir(cur)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Experiment: {EXP_TAG} (AGE_MODE={AGE_MODE}) | Working dir: {os.getcwd()}")
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# [Cell 2] GIAI ĐOẠN 1 — Chuẩn bị & Sửa dữ liệu (PadToSquare AAF + Resolution Jitter AFAD + Domain Aug FairFace + 2-Stage Sampler)
import pandas as pd
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

AGE_GROUPS_9 = ["0-2", "3-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]
AGE_GROUP_TO_IDX = {g: i for i, g in enumerate(AGE_GROUPS_9)}
AGE_CUTOFFS_8 = [2.5, 9.5, 19.5, 29.5, 39.5, 49.5, 59.5, 69.5]

with open("dataset/metadata/norm_stats.json", "r", encoding="utf-8") as f:
    norm_stats = json.load(f)
TRAIN_MEAN, TRAIN_STD = norm_stats["mean"], norm_stats["std"]
FILL_RGB = tuple(int(round(m * 255)) for m in TRAIN_MEAN)

class PadToSquare:
    """Đệm ảnh chữ nhật (AAF 309x398) về hình vuông bằng màu trung bình tập Train trước khi Resize."""
    def __init__(self, fill=FILL_RGB):
        self.fill = tuple(int(c) for c in fill)
    def __call__(self, img):
        w, h = img.size
        if w == h:
            return img
        side = max(w, h)
        pad_l = (side - w) // 2
        pad_t = (side - h) // 2
        return ImageOps.expand(img, border=(pad_l, pad_t, side - w - pad_l, side - h - pad_t), fill=self.fill)

class RandomDownscaleUpscale:
    """Mô phỏng độ phân giải thấp (AFAD-Full 40-61px) để chống học đường tắt theo độ mờ."""
    def __init__(self, min_size=48, max_size=112, p=0.2):
        self.min_size, self.max_size, self.p = min_size, max_size, p
    def __call__(self, img):
        if random.random() >= self.p:
            return img
        w, h = img.size
        low = random.randint(self.min_size, self.max_size)
        return img.resize((low, low), Image.BILINEAR).resize((w, h), Image.BILINEAR)

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 0  # An toàn cho Python 3.13 trên Google Colab

train_transform = transforms.Compose([
    PadToSquare(fill=FILL_RGB),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.82, 1.0), ratio=(0.92, 1.08)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.08, 0.08), scale=(0.92, 1.08)),
    transforms.RandomPerspective(distortion_scale=0.12, p=0.25),
    RandomDownscaleUpscale(min_size=48, max_size=112, p=0.2),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=TRAIN_MEAN, std=TRAIN_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.10), value=0),
])

val_transform = transforms.Compose([
    PadToSquare(fill=FILL_RGB),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=TRAIN_MEAN, std=TRAIN_STD),
])

class AsianFaceUltimateDataset(Dataset):
    """Trả về (img, gender, age_grp, age_exact, has_exact) để hỗ trợ Soft Ordinal Labels (LDL)."""
    def __init__(self, csv_path, transform=None, img_root_override=None):
        self.df = pd.read_csv(csv_path)
        if img_root_override:
            self.df["image_path"] = self.df["image_path"].astype(str).str.replace("dataset/processed", img_root_override, regex=False)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(str(row["image_path"]).replace("\\", "/")).convert("RGB")
        if self.transform:
            img = self.transform(img)
        gender = torch.tensor(float(row["gender"]), dtype=torch.float32)
        age_grp = torch.tensor(AGE_GROUP_TO_IDX[str(row["age_group"]).strip()], dtype=torch.long)
        age_exact = torch.tensor(float(row["age"]), dtype=torch.float32)
        has_exact = torch.tensor(0.0 if str(row["source"]).strip() == "FairFace" else 1.0, dtype=torch.float32)
        return img, gender, age_grp, age_exact, has_exact

train_ds = AsianFaceUltimateDataset("dataset/metadata/train.csv", transform=train_transform, img_root_override=IMG_ROOT_OVERRIDE)
val_ds = AsianFaceUltimateDataset("dataset/metadata/val.csv", transform=val_transform, img_root_override=IMG_ROOT_OVERRIDE)
test_ds = AsianFaceUltimateDataset("dataset/metadata/test.csv", transform=val_transform, img_root_override=IMG_ROOT_OVERRIDE)

# Lấy mẫu 2 Pha:
# - Pha 1 (Epoch 1-32): Half-Balanced (1 / sqrt(count)) — chống lặp mẫu 60-69 quá đà
# - Pha 2 (Epoch 33-45): Phân bố tự nhiên (shuffle=True) — triệt tiêu Prior Shift với Val/Test
strat_key = train_ds.df["age_group"].astype(str) + "_" + train_ds.df["gender"].astype(str)
counts = strat_key.value_counts().to_dict()
half_weights = strat_key.map(lambda k: 1.0 / np.sqrt(counts[k])).values
sampler_stage1 = WeightedRandomSampler(weights=torch.DoubleTensor(half_weights), num_samples=len(half_weights), replacement=True)

train_loader_stage1 = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler_stage1, num_workers=NUM_WORKERS, pin_memory=True)
train_loader_stage2 = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Dataset loaded -> Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")
print(f"PadToSquare fill RGB: {FILL_RGB} | Exact-age ratio in Train: {(train_ds.df['source'] != 'FairFace').mean()*100:.1f}%")

In [ ]:
# [Cell 3] GIAI ĐOẠN 2 — Kiến trúc ResNet18HybridUltimate (DeepStem + ResNet-D + CBAM + MHSA 7x7 + Multi-Scale + Gender->Age)
import torch.nn as nn
import torch.nn.functional as F

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, mid, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, kernel_size=1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction=reduction)
        self.sa = SpatialAttention(kernel_size=kernel_size)

    def forward(self, x):
        return self.sa(self.ca(x))


class BasicBlockD_CBAM(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.cbam = CBAMBlock(out_channels, reduction=16, kernel_size=7)

        if stride != 1 or in_channels != out_channels * self.expansion:
            layers = []
            if stride != 1:
                layers.append(nn.AvgPool2d(kernel_size=stride, stride=stride, ceil_mode=True))
            layers.extend([
                nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1, stride=1, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion),
            ])
            self.shortcut = nn.Sequential(*layers)
        else:
            self.shortcut = nn.Sequential()

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.cbam(self.bn2(self.conv2(out)))
        out += self.shortcut(x)
        return self.relu(out)


class SpatialMHSA7x7(nn.Module):
    """Multi-Head Self-Attention (4 heads) trên 7x7 = 49 tokens cuối layer4 cho ngữ cảnh hình học toàn mặt."""
    def __init__(self, dim=512, num_heads=4, dropout=0.1, spatial_size=7):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.zeros(1, spatial_size * spatial_size, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        b, c, h, w = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        if tokens.size(1) == self.pos_embed.size(1):
            tokens = tokens + self.pos_embed
        t1 = self.norm1(tokens)
        attn_out, _ = self.attn(t1, t1, t1, need_weights=False)
        tokens = tokens + attn_out
        tokens = tokens + self.ffn(self.norm2(tokens))
        return tokens.transpose(1, 2).reshape(b, c, h, w)


class ResNet18HybridUltimate(nn.Module):
    def __init__(self, num_age_groups=9, dropout=0.3):
        super().__init__()
        self.in_channels = 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, num_blocks=2, stride=1)
        self.layer2 = self._make_layer(128, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(256, num_blocks=2, stride=2)
        self.layer4 = self._make_layer(512, num_blocks=2, stride=2)
        self.mhsa = SpatialMHSA7x7(dim=512, num_heads=4, dropout=0.1, spatial_size=7)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Nhánh Giới tính
        self.gender_proj = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(512, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.GELU(),
        )
        self.gender_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

        # Cross-Task Conditioning: ghép gender_proj (256-d, detach) + Multi-Scale (768-d) = 1024-d vào Age Head
        self.age_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256 + 512 + 256, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_age_groups - 1),
        )
        self._init_weights()

    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlockD_CBAM(self.in_channels, out_channels, stride=s))
            self.in_channels = out_channels * BasicBlockD_CBAM.expansion
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def extract_multiscale(self, x):
        x = self.maxpool(self.stem(x))
        x = self.layer1(x)
        x = self.layer2(x)
        f3 = self.layer3(x)
        f4 = self.mhsa(self.layer4(f3))
        feat3 = self.avgpool(f3).flatten(1)
        feat4 = self.avgpool(f4).flatten(1)
        return feat4, torch.cat([feat3, feat4], dim=1)

    def extract_features(self, x):
        _, feat34 = self.extract_multiscale(x)
        return feat34

    def forward(self, x):
        feat_g, feat_a = self.extract_multiscale(x)
        g_emb = self.gender_proj(feat_g)
        g_logit = self.gender_head(g_emb).squeeze(1)
        a_logits = self.age_head(torch.cat([feat_a, g_emb.detach()], dim=1))
        return g_logit, a_logits

    @torch.no_grad()
    def predict_proba(self, x, tta=True, enforce_monotonic=True, thresholds=None):
        g_logit, a_logits = self.forward(x)
        g_prob_male = torch.sigmoid(g_logit)
        a_scores = torch.sigmoid(a_logits)
        if tta:
            g_logit_f, a_logits_f = self.forward(torch.flip(x, dims=[3]))
            g_prob_male = 0.5 * (g_prob_male + torch.sigmoid(g_logit_f))
            a_scores = 0.5 * (a_scores + torch.sigmoid(a_logits_f))
        if enforce_monotonic:
            a_scores = torch.cummin(a_scores, dim=1).values
        if thresholds is not None:
            th = torch.as_tensor(thresholds, device=a_scores.device, dtype=a_scores.dtype).unsqueeze(0)
            a_pred_grp = (a_scores > th).sum(dim=1).long()
        else:
            a_pred_grp = (a_scores > 0.5).sum(dim=1).long()
        return g_prob_male, a_pred_grp, a_scores

model = ResNet18HybridUltimate(num_age_groups=9, dropout=0.3).to(device)
print(f"Model: {EXP_TAG} (DeepStem + CBAM + MHSA7x7 + MultiScale + Gender->Age) | Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

In [ ]:
# [Cell 4] Soft Ordinal Labels (LDL) + Kendall Uncertainty Weighting + Model EMA + Warm-up 5 Epochs
NUM_EPOCHS = 45
STAGE1_EPOCHS = 32     # Epoch 1-32: Half-Balanced Sampler | Epoch 33-45: Natural Distribution Fine-tune
WARMUP_EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.1
LDL_TAU = 1.5          # Độ mượt ranh giới tuổi thực (năm) cho AFAD-Full/AAF/UTKFace
FAIRFACE_SMOOTH = 0.03
MONO_LAMBDA = 0.25
GRAD_CLIP_NORM = 1.0
EMA_DECAY = 0.995
EARLY_STOP_PATIENCE = 14

class UltimateMultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce_gender = nn.BCEWithLogitsLoss()
        self.register_buffer("threshold_weights", torch.tensor([1.0, 1.0, 1.4, 1.4, 1.15, 1.0, 1.0, 1.0]))
        self.register_buffer("age_cutoffs", torch.tensor(AGE_CUTOFFS_8, dtype=torch.float32))
        # Tham số Kendall Uncertainty Weighting (clamp [-4, 4] chống bùng nổ)
        self.log_var_gender = nn.Parameter(torch.tensor(0.0))
        self.log_var_age = nn.Parameter(torch.tensor(-0.5))

    def forward(self, g_logit, a_logits, g_true, a_grp_true, age_exact, has_exact):
        g_target = g_true.float() * (1.0 - LABEL_SMOOTHING) + 0.5 * LABEL_SMOOTHING
        lg = self.bce_gender(g_logit, g_target)

        thresholds = torch.arange(8, device=a_logits.device).unsqueeze(0)
        hard_levels = (a_grp_true.unsqueeze(1) > thresholds).float()
        hard_smooth = hard_levels * (1.0 - 2.0 * FAIRFACE_SMOOTH) + FAIRFACE_SMOOTH
        soft_exact = torch.sigmoid((age_exact.float().unsqueeze(1) - self.age_cutoffs.unsqueeze(0)) / LDL_TAU)
        mask = has_exact.float().unsqueeze(1)
        targets = mask * soft_exact + (1.0 - mask) * hard_smooth

        bce_mat = F.binary_cross_entropy_with_logits(a_logits, targets, reduction="none")
        w = self.threshold_weights.unsqueeze(0)
        la_base = (bce_mat * w).sum() / (w.sum() * a_logits.size(0))
        probs = torch.sigmoid(a_logits)
        la = la_base + MONO_LAMBDA * torch.relu(probs[:, 1:] - probs[:, :-1]).mean()

        sg = torch.clamp(self.log_var_gender, -4.0, 4.0)
        sa = torch.clamp(self.log_var_age, -4.0, 4.0)
        total = 0.5 * torch.exp(-sg) * lg + 0.5 * sg + 0.5 * torch.exp(-sa) * la + 0.5 * sa
        return total, lg, la

class ModelEMA:
    """Exponential Moving Average trọng số + đồng bộ BatchNorm buffers giúp chống overfit cuối chu kỳ."""
    def __init__(self, model, decay=0.995):
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        for ep, mp in zip(self.ema_model.parameters(), model.parameters()):
            ep.data.mul_(self.decay).add_(mp.data, alpha=1.0 - self.decay)
        for eb, mb in zip(self.ema_model.buffers(), model.buffers()):
            eb.data.copy_(mb.data)

def decode_age_group(a_logits):
    probs = torch.cummin(torch.sigmoid(a_logits), dim=1).values
    return (probs > 0.5).sum(dim=1).long()

criterion = UltimateMultiTaskLoss().to(device)
ema = ModelEMA(model, decay=EMA_DECAY)
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(criterion.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.05, total_iters=WARMUP_EPOCHS)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS])
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
print(f"Loss: Soft-Ordinal LDL(tau={LDL_TAU}) + Kendall Uncertainty + EMA({EMA_DECAY}) | Epochs={NUM_EPOCHS} (Stage1={STAGE1_EPOCHS})")

In [ ]:
# [Cell 5] Vòng lặp Huấn luyện 45 Epochs (2-Stage Sampling + EMA + Lưu song song best_loss & best_acc)
from tqdm.auto import tqdm

CKPT_DIR = "checkpoints"
OUT_DIR = f"results/{EXP_TAG}"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
BEST_LOSS_PATH = os.path.join(CKPT_DIR, f"{EXP_TAG}_best_loss.pth")
BEST_ACC_PATH = os.path.join(CKPT_DIR, f"{EXP_TAG}_best_acc.pth")

history = []
best_val_loss = float("inf")
best_combo_score = 0.0
best_acc_epoch = 0
epochs_no_improve = 0
t_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    active_loader = train_loader_stage1 if epoch <= STAGE1_EPOCHS else train_loader_stage2
    stage_lbl = "S1-HalfBal" if epoch <= STAGE1_EPOCHS else "S2-Natural"

    model.train()
    tr_loss, tr_g_loss, tr_a_loss = 0.0, 0.0, 0.0
    tr_g_ok, tr_a_ok, tr_a_adj_ok, tr_n = 0, 0, 0, 0

    for imgs, genders, age_grps, age_exact, has_exact in tqdm(active_loader, desc=f"Ep {epoch:02d}/{NUM_EPOCHS} [{stage_lbl}]", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        genders = genders.to(device, non_blocking=True)
        age_grps = age_grps.to(device, non_blocking=True)
        age_exact = age_exact.to(device, non_blocking=True)
        has_exact = has_exact.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            g_logit, a_logits = model(imgs)
            loss, lg, la = criterion(g_logit, a_logits, genders, age_grps, age_exact, has_exact)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        bs = imgs.size(0)
        tr_loss += loss.item() * bs
        tr_g_loss += lg.item() * bs
        tr_a_loss += la.item() * bs
        g_pred = (torch.sigmoid(g_logit) >= 0.5).long()
        a_pred = decode_age_group(a_logits)
        tr_g_ok += (g_pred == genders.long()).sum().item()
        tr_a_ok += (a_pred == age_grps).sum().item()
        tr_a_adj_ok += (torch.abs(a_pred - age_grps) <= 1).sum().item()
        tr_n += bs

    # Đánh giá trên tập Validation bằng trọng số EMA (ổn định hơn ở cuối chu kỳ)
    eval_model = ema.ema_model if epoch >= WARMUP_EPOCHS else model
    eval_model.eval()
    va_loss, va_g_loss, va_a_loss = 0.0, 0.0, 0.0
    va_g_ok, va_a_ok, va_a_adj_ok, va_n = 0, 0, 0, 0
    with torch.no_grad():
        for imgs, genders, age_grps, age_exact, has_exact in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            genders = genders.to(device, non_blocking=True)
            age_grps = age_grps.to(device, non_blocking=True)
            age_exact = age_exact.to(device, non_blocking=True)
            has_exact = has_exact.to(device, non_blocking=True)

            g_logit, a_logits = eval_model(imgs)
            loss, lg, la = criterion(g_logit, a_logits, genders, age_grps, age_exact, has_exact)

            bs = imgs.size(0)
            va_loss += loss.item() * bs
            va_g_loss += lg.item() * bs
            va_a_loss += la.item() * bs
            g_pred = (torch.sigmoid(g_logit) >= 0.5).long()
            a_pred = decode_age_group(a_logits)
            va_g_ok += (g_pred == genders.long()).sum().item()
            va_a_ok += (a_pred == age_grps).sum().item()
            va_a_adj_ok += (torch.abs(a_pred - age_grps) <= 1).sum().item()
            va_n += bs

    cur_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    va_acc = va_g_ok / va_n
    va_age_acc = va_a_ok / va_n
    combo_score = 0.4 * va_acc + 0.6 * va_age_acc
    wg_eff = float(torch.exp(-torch.clamp(criterion.log_var_gender, -4.0, 4.0)).item())
    wa_eff = float(torch.exp(-torch.clamp(criterion.log_var_age, -4.0, 4.0)).item())

    row = {
        "epoch": epoch, "stage": stage_lbl, "lr": cur_lr, "w_gender_eff": wg_eff, "w_age_eff": wa_eff,
        "tr_loss": tr_loss / tr_n, "tr_gender_loss": tr_g_loss / tr_n, "tr_age_loss": tr_a_loss / tr_n,
        "tr_acc": tr_g_ok / tr_n, "tr_age_grp_acc": tr_a_ok / tr_n, "tr_age_adj_acc": tr_a_adj_ok / tr_n,
        "val_loss": va_loss / va_n, "val_gender_loss": va_g_loss / va_n, "val_age_loss": va_a_loss / va_n,
        "val_acc": va_acc, "val_age_grp_acc": va_age_acc, "val_age_adj_acc": va_a_adj_ok / va_n,
        "val_combo": combo_score,
    }
    history.append(row)
    print(f"Ep [{epoch:02d}/{NUM_EPOCHS}][{stage_lbl}] (lr={cur_lr:.1e}, wg={wg_eff:.2f}, wa={wa_eff:.2f}) | "
          f"Tr G:{row['tr_acc']*100:.1f}% A:{row['tr_age_grp_acc']*100:.1f}% | "
          f"Val G:{va_acc*100:.1f}% A:{va_age_acc*100:.1f}% (Adj±1:{row['val_age_adj_acc']*100:.1f}%) Combo:{combo_score*100:.2f}%")

    if row["val_loss"] < best_val_loss:
        best_val_loss = row["val_loss"]
        torch.save({"epoch": epoch, "age_mode": AGE_MODE, "model_state_dict": eval_model.state_dict(), "val_loss": best_val_loss}, BEST_LOSS_PATH)

    if combo_score > best_combo_score:
        best_combo_score = combo_score
        best_acc_epoch = epoch
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "age_mode": AGE_MODE,
            "model_state_dict": eval_model.state_dict(),
            "val_combo": best_combo_score,
            "val_acc": va_acc,
            "val_age_grp_acc": va_age_acc,
        }, BEST_ACC_PATH)
    else:
        epochs_no_improve += 1
        if epoch > STAGE1_EPOCHS + 3 and epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early Stopping tại epoch {epoch} (không cải thiện Combo Score sau {EARLY_STOP_PATIENCE} epochs).")
            break

torch.save({"epoch": len(history), "age_mode": AGE_MODE, "model_state_dict": ema.ema_model.state_dict()}, os.path.join(CKPT_DIR, f"{EXP_TAG}_last.pth"))
pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
if DRIVE_SAVE_DIR:
    shutil.copytree(CKPT_DIR, os.path.join(DRIVE_SAVE_DIR, "checkpoints"), dirs_exist_ok=True)
    shutil.copytree("results", os.path.join(DRIVE_SAVE_DIR, "results"), dirs_exist_ok=True)
print(f"Hoàn tất ({len(history)} epochs) trong {(time.time()-t_start)/60:.1f} phút! Best Combo Epoch: {best_acc_epoch} ({best_combo_score*100:.2f}%)")

In [ ]:
# [Cell 6] Biểu đồ LOSS & ACCURACY của Gender và Age (đánh dấu điểm chuyển Pha 2 ở Epoch 32)
import matplotlib.pyplot as plt

hist_df = pd.read_csv(os.path.join(OUT_DIR, "history.csv"))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, cols, title in zip(
    axes,
    [("tr_gender_loss", "val_gender_loss"), ("tr_age_loss", "val_age_loss"), ("tr_loss", "val_loss")],
    ["Gender Loss (BCE + Label Smoothing)", "Age Loss (Soft-Ordinal LDL + Mono)", "Total Kendall Uncertainty Loss"],
):
    ax.plot(hist_df["epoch"], hist_df[cols[0]], marker="o", markersize=3, label="Train")
    ax.plot(hist_df["epoch"], hist_df[cols[1]], marker="s", markersize=3, label="Validation (EMA)")
    ax.axvline(STAGE1_EPOCHS + 0.5, color="gray", linestyle="--", alpha=0.6, label="Stage 2 (Natural)")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle(f"Biểu đồ Loss của Gender và Age ({EXP_TAG} — {len(hist_df)} Epochs)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "gender_age_loss_curves.png"), dpi=150); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, cols, title in zip(
    axes,
    [("tr_acc", "val_acc"), ("tr_age_grp_acc", "val_age_grp_acc"), ("tr_age_adj_acc", "val_age_adj_acc")],
    ["Gender Accuracy", "Age-Group Accuracy (9 nhóm tuổi)", "Age Adjacent Accuracy (Dung sai ±1 nhóm)"],
):
    ax.plot(hist_df["epoch"], hist_df[cols[0]] * 100, marker="o", markersize=3, label="Train")
    ax.plot(hist_df["epoch"], hist_df[cols[1]] * 100, marker="s", markersize=3, label="Validation (EMA)")
    ax.axvline(STAGE1_EPOCHS + 0.5, color="gray", linestyle="--", alpha=0.6)
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle(f"Biểu đồ Accuracy của Gender và Age ({EXP_TAG} — {len(hist_df)} Epochs)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "gender_age_accuracy_curves.png"), dpi=150); plt.show()

In [ ]:
# [Cell 7] Hiệu chỉnh 8 Ngưỡng CORAL trên tập Val + Đánh giá Test (Single vs TTA vs TTA+Calibrated) + 5 Biểu đồ
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

ckpt = torch.load(BEST_ACC_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded Best Combo EMA Checkpoint (Epoch {ckpt.get('epoch')}) from {BEST_ACC_PATH}")

def run_loader_pass(loader, use_tta, extract_feats=False):
    g_probs, g_trues, a_trues, a_scores, feats_list = [], [], [], [], []
    t0 = time.time()
    with torch.no_grad():
        for imgs, genders, age_grps, _, _ in loader:
            imgs_dev = imgs.to(device)
            if extract_feats:
                feats_list.append(model.extract_features(imgs_dev).cpu().numpy())
            gp, _, asc = model.predict_proba(imgs_dev, tta=use_tta, enforce_monotonic=True)
            g_probs.append(gp.cpu().numpy())
            g_trues.append(genders.numpy())
            a_trues.append(age_grps.numpy())
            a_scores.append(asc.cpu().numpy())
    infer_ms = (time.time() - t0) * 1000.0 / len(loader.dataset)
    return (
        np.concatenate(g_probs),
        np.concatenate(g_trues).astype(int),
        np.concatenate(a_trues).astype(int),
        np.concatenate(a_scores),
        np.concatenate(feats_list) if feats_list else None,
        infer_ms,
    )

# 1. Hiệu chỉnh 8 ngưỡng CORAL trên tập Validation (khắc phục Prior Shift mà không đụng vào Test)
_, _, val_at, val_asc_tta, _, _ = run_loader_pass(val_loader, use_tta=True, extract_feats=False)
cal_thresholds = np.full(8, 0.5, dtype=np.float32)
best_val_cal_acc = accuracy_score(val_at, (val_asc_tta > cal_thresholds).sum(axis=1))
for _ in range(3):
    for k in range(8):
        best_t = cal_thresholds[k]
        for cand in np.linspace(0.35, 0.65, 31):
            tmp = cal_thresholds.copy()
            tmp[k] = cand
            acc = accuracy_score(val_at, (val_asc_tta > tmp).sum(axis=1))
            if acc > best_val_cal_acc + 1e-6:
                best_val_cal_acc = acc
                best_t = float(cand)
        cal_thresholds[k] = best_t
print("Val-Calibrated 8 CORAL Thresholds:", np.round(cal_thresholds, 3).tolist(), f"| Val Exact Acc: {best_val_cal_acc*100:.2f}%")

# 2. Đánh giá trên tập Test
gp_sv, gt, at, asc_sv, feats_np, ms_sv = run_loader_pass(test_loader, use_tta=False, extract_feats=True)
gp_tta, _, _, asc_tta, _, ms_tta = run_loader_pass(test_loader, use_tta=True, extract_feats=False)

ap_sv = (asc_sv > 0.5).sum(axis=1)
ap_tta = (asc_tta > 0.5).sum(axis=1)
ap_cal = (asc_tta > cal_thresholds[None, :]).sum(axis=1)

def build_metrics_dict(gp, ap, infer_ms, mode_name):
    g_pred = (gp >= 0.5).astype(int)
    return {
        "mode": mode_name,
        "gender_acc": float(accuracy_score(gt, g_pred)),
        "gender_precision": float(precision_score(gt, g_pred, average="macro", zero_division=0)),
        "gender_recall": float(recall_score(gt, g_pred, average="macro", zero_division=0)),
        "gender_f1": float(f1_score(gt, g_pred, average="macro", zero_division=0)),
        "gender_roc_auc": float(roc_auc_score(gt, gp)),
        "age_group_acc": float(accuracy_score(at, ap)),
        "age_adjacent_acc (±1 group)": float(np.mean(np.abs(ap - at) <= 1)),
        "age_precision": float(precision_score(at, ap, average="macro", zero_division=0)),
        "age_recall": float(recall_score(at, ap, average="macro", zero_division=0)),
        "age_f1": float(f1_score(at, ap, average="macro", zero_division=0)),
        "inference_ms_per_img": round(infer_ms, 3),
    }

m_sv = build_metrics_dict(gp_sv, ap_sv, ms_sv, "Single-View (0.50)")
m_tta = build_metrics_dict(gp_tta, ap_tta, ms_tta, "TTA (0.50)")
m_cal = build_metrics_dict(gp_tta, ap_cal, ms_tta, "TTA + Val-Calibrated")
m_cal.update({
    "experiment": EXP_TAG,
    "age_mode": AGE_MODE,
    "best_epoch": int(ckpt.get("epoch", 0)),
    "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    "calibrated_thresholds": [round(float(x), 4) for x in cal_thresholds],
})

with open(os.path.join(OUT_DIR, "test_metrics.json"), "w", encoding="utf-8") as f:
    json.dump({"single_view": m_sv, "tta": m_tta, "tta_calibrated": m_cal}, f, indent=2)

# Chọn bản tốt nhất giữa TTA (0.50) và TTA + Calibrated để vẽ Ma trận nhầm lẫn
use_cal = m_cal["age_group_acc"] >= m_tta["age_group_acc"]
ACTIVE_THRESHOLDS = cal_thresholds if use_cal else np.full(8, 0.5, dtype=np.float32)
gender_prob_male = gp_tta
gender_true = gt
gender_pred = (gender_prob_male >= 0.5).astype(int)
age_grp_true = at
age_grp_pred = ap_cal if use_cal else ap_tta
coral_scores_tta = asc_tta

display(pd.DataFrame([
    {
        "Chế độ đánh giá": m["mode"],
        "Gender Acc": f"{m['gender_acc']*100:.2f}%",
        "ROC-AUC": f"{m['gender_roc_auc']*100:.2f}%",
        "Age Exact Acc (9 nhóm)": f"{m['age_group_acc']*100:.2f}%",
        "Age Adjacent (±1 nhóm)": f"{m['age_adjacent_acc (±1 group)']*100:.2f}%",
        "Age F1": f"{m['age_f1']*100:.2f}%",
    }
    for m in (m_sv, m_tta, m_cal)
]))

cm_gender = confusion_matrix(gender_true, gender_pred, labels=[0, 1])
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm_gender, annot=True, fmt="d", cmap="Blues", xticklabels=["Female (0)", "Male (1)"], yticklabels=["Female (0)", "Male (1)"])
plt.title(f"Ma trận nhầm lẫn Giới tính — TTA ({EXP_TAG})"); plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "gender_confusion_matrix.png"), dpi=150); plt.show()

cm_age = confusion_matrix(age_grp_true, age_grp_pred, labels=list(range(len(AGE_GROUPS_9))))
plt.figure(figsize=(9, 7.5))
sns.heatmap(cm_age, annot=True, fmt="d", cmap="YlGnBu", xticklabels=AGE_GROUPS_9, yticklabels=AGE_GROUPS_9)
plt.title(f"Ma trận nhầm lẫn Độ tuổi 9 nhóm ({EXP_TAG})"); plt.xlabel("Predicted Age Group"); plt.ylabel("True Age Group")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "age_confusion_matrix.png"), dpi=150); plt.show()

X_pca = PCA(n_components=50, random_state=42).fit_transform(feats_np)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_pca)
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
cmap_age = plt.colormaps["tab10"]
for idx, ag in enumerate(AGE_GROUPS_9):
    m = age_grp_true == idx
    axes[0].scatter(X_tsne[m, 0], X_tsne[m, 1], s=12, alpha=0.7, label=ag, color=cmap_age(idx))
axes[0].set_title(f"t-SNE không gian đặc trưng Hybrid 768-d theo 9 Nhóm tuổi ({EXP_TAG})")
axes[0].set_xlabel("t-SNE 1"); axes[0].set_ylabel("t-SNE 2"); axes[0].legend(title="Nhóm tuổi", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
for g_lbl, name, col in [(0, "Female (0)", "#db2777"), (1, "Male (1)", "#1d4ed8")]:
    m = gender_true == g_lbl
    axes[1].scatter(X_tsne[m, 0], X_tsne[m, 1], s=12, alpha=0.65, label=name, color=col)
axes[1].set_title(f"t-SNE không gian đặc trưng Hybrid 768-d theo Giới tính ({EXP_TAG})")
axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2"); axes[1].legend(title="Giới tính")
plt.suptitle(f"Biểu diễn không gian đặc trưng Hybrid (768-d) trên tập Test ({EXP_TAG})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "tsne_age_features.png"), dpi=150, bbox_inches="tight"); plt.show()

if DRIVE_SAVE_DIR:
    shutil.copytree("results", os.path.join(DRIVE_SAVE_DIR, "results"), dirs_exist_ok=True)
    print("Đã đồng bộ toàn bộ biểu đồ & test_metrics.json lên Google Drive:", DRIVE_SAVE_DIR)

In [ ]:
# [Cell 8] Chẩn đoán chi tiết: Theo Nhóm tuổi, Theo Nguồn (kiểm tra AAF & FairFace) & 8 Ngưỡng CORAL
test_meta = test_ds.df.copy()
test_meta["p_male_sigmoid"] = np.round(gender_prob_male, 4)
test_meta["pred_gender_idx"] = gender_pred
test_meta["gender_ok"] = (gender_pred == gender_true).astype(int)
test_meta["true_age_grp_idx"] = age_grp_true
test_meta["pred_age_grp_idx"] = age_grp_pred
test_meta["pred_age_group"] = [AGE_GROUPS_9[i] for i in age_grp_pred]
test_meta["age_exact_correct"] = (age_grp_pred == age_grp_true).astype(int)
test_meta["age_adj_correct"] = (np.abs(age_grp_pred - age_grp_true) <= 1).astype(int)

print("=== 1. ĐỘ CHÍNH XÁC THEO TỪNG NHÓM TUỔI ===")
display(test_meta.groupby("age_group")[["gender_ok", "age_exact_correct", "age_adj_correct"]].agg(
    count=("age_exact_correct", "count"),
    gender_acc_pct=("gender_ok", lambda s: round(s.mean() * 100, 2)),
    exact_acc_pct=("age_exact_correct", lambda s: round(s.mean() * 100, 2)),
    adjacent_acc_pct=("age_adj_correct", lambda s: round(s.mean() * 100, 2)),
).reindex(AGE_GROUPS_9))

print("\n=== 2. ĐỘ CHÍNH XÁC THEO NGUỒN DỮ LIỆU (Kiểm tra tác dụng PadToSquare trên AAF & MHSA trên FairFace) ===")
display(test_meta.groupby("source")[["gender_ok", "age_exact_correct", "age_adj_correct"]].agg(
    count=("age_exact_correct", "count"),
    gender_acc_pct=("gender_ok", lambda s: round(s.mean() * 100, 2)),
    exact_acc_pct=("age_exact_correct", lambda s: round(s.mean() * 100, 2)),
    adjacent_acc_pct=("age_adj_correct", lambda s: round(s.mean() * 100, 2)),
))

print("\n=== 3. ĐỘ CHÍNH XÁC 8 NGƯỠNG CORAL & KIỂM TRA TÍNH ĐƠN ĐIỆU ===")
for k in range(8):
    tgt = (age_grp_true > k).astype(int)
    pk = (coral_scores_tta[:, k] > ACTIVE_THRESHOLDS[k]).astype(int)
    print(f"Ngưỡng > {AGE_GROUPS_9[k]:>5s} (t={ACTIVE_THRESHOLDS[k]:.2f}): Acc = {(pk == tgt).mean()*100:.2f}%")
non_mono = np.any(np.diff(coral_scores_tta, axis=1) > 1e-4, axis=1).mean() * 100
print(f"Tỷ lệ vi phạm đơn điệu CORAL sau Isotonic Clipping: {non_mono:.2f}% (Kỳ vọng: 0.00%)")

sample_rows = test_meta.sample(6, random_state=42)
fig, axes = plt.subplots(1, 6, figsize=(17, 3.8))
for ax, row in zip(axes, sample_rows.itertuples()):
    with Image.open(str(row.image_path).replace("\\", "/")) as im:
        ax.imshow(im.convert("RGB"))
    g_true = "Female" if row.gender == 0 else "Male"
    g_pred = "Female" if row.pred_gender_idx == 0 else "Male"
    ax.set_title(f"True: {g_true} | {row.age_group}\nPred: {g_pred} | {row.pred_age_group}\nP(Male)={row.p_male_sigmoid:.2f}", fontsize=8.5)
    ax.axis("off")
plt.suptitle(f"Dự đoán mẫu trên tập Test — TTA ({EXP_TAG})", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# [Cell 9] Tải ảnh từ Internet (URL) hoặc tải file từ máy lên để Dự đoán bằng TTA + Calibrated Thresholds
import io, urllib.request
import cv2

def load_image_from_url_or_path(url="", local_path="", upload_on_colab=False):
    if upload_on_colab and IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            fname = next(iter(uploaded.keys()))
            return Image.open(io.BytesIO(uploaded[fname])).convert("RGB"), f"Uploaded: {fname}"
    if url and url.strip():
        req = urllib.request.Request(url.strip(), headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            return Image.open(io.BytesIO(resp.read())).convert("RGB"), f"URL: {url.strip()[:50]}..."
    if local_path and os.path.exists(local_path):
        return Image.open(local_path).convert("RGB"), f"File: {os.path.basename(local_path)}"
    fallback_path = str(test_ds.df.iloc[0]["image_path"]).replace("\\", "/")
    return Image.open(fallback_path).convert("RGB"), f"Test Sample: {os.path.basename(fallback_path)}"

def detect_and_crop_face(pil_img, margin_ratio=0.25):
    img_np = np.array(pil_img)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    cascade_path = os.path.join(cv2.data.haarcascades, "haarcascade_frontalface_default.xml")
    face_cascade = cv2.CascadeClassifier(cascade_path)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(48, 48))
    if len(faces) == 0:
        return pil_img, None
    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
    H, W = img_np.shape[:2]
    mx, my = int(w * margin_ratio), int(h * margin_ratio)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(W, x + w + mx), min(H, y + h + my)
    return pil_img.crop((x1, y1, x2, y2)), (x1, y1, x2, y2)

def predict_single_image(url="", local_path="", upload_on_colab=False, auto_crop_face=True, use_tta=True):
    raw_img, src_label = load_image_from_url_or_path(url=url, local_path=local_path, upload_on_colab=upload_on_colab)
    face_img, bbox = detect_and_crop_face(raw_img) if auto_crop_face else (raw_img, None)

    model.eval()
    x_tensor = val_transform(face_img).unsqueeze(0).to(device)
    with torch.no_grad():
        g_prob_male_t, a_pred_grp_t, a_scores_t = model.predict_proba(
            x_tensor, tta=use_tta, enforce_monotonic=True, thresholds=ACTIVE_THRESHOLDS
        )

    p_male = float(g_prob_male_t.item())
    p_female = 1.0 - p_male
    pred_gender_str = "Male (Nam - 1)" if p_male >= 0.5 else "Female (Nữ - 0)"
    pred_age_idx = int(a_pred_grp_t.item())
    pred_age_group = AGE_GROUPS_9[pred_age_idx]

    p_gt = np.clip(a_scores_t.squeeze(0).cpu().numpy(), 0.0, 1.0)
    probs_9 = np.zeros(9, dtype=np.float32)
    probs_9[0] = 1.0 - p_gt[0]
    for k in range(1, 8):
        probs_9[k] = max(0.0, p_gt[k - 1] - p_gt[k])
    probs_9[8] = p_gt[7]
    if probs_9.sum() > 0:
        probs_9 = probs_9 / probs_9.sum()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1, 1, 1.4]})
    img_draw = np.array(raw_img).copy()
    if bbox is not None:
        cv2.rectangle(img_draw, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), max(2, img_draw.shape[1] // 150))
    axes[0].imshow(img_draw)
    axes[0].set_title(f"Ảnh gốc ({'Đã cắt mặt' if bbox else 'Ảnh gốc'})\n{src_label}", fontsize=9)
    axes[0].axis("off")

    axes[1].imshow(PadToSquare(FILL_RGB)(face_img).resize((IMG_SIZE, IMG_SIZE)))
    axes[1].set_title(
        f"Dự đoán ({EXP_TAG}, TTA={use_tta})\n"
        f"Giới tính: {pred_gender_str}\n"
        f"P(Male)={p_male:.3f} | P(Female)={p_female:.3f}\n"
        f"Nhóm tuổi: {pred_age_group}",
        fontsize=9.5,
        fontweight="bold",
    )
    axes[1].axis("off")

    colors = ["#1d4ed8" if i == pred_age_idx else "#93c5fd" for i in range(9)]
    axes[2].barh(AGE_GROUPS_9, probs_9 * 100.0, color=colors)
    axes[2].set_xlim(0, 100)
    axes[2].set_xlabel("Xác suất (%)")
    axes[2].set_title("Phân bố dự đoán 9 Nhóm tuổi (Soft-Ordinal + Isotonic)", fontsize=9.5)
    axes[2].grid(axis="x", alpha=0.3)
    for i, v in enumerate(probs_9 * 100.0):
        axes[2].text(min(v + 1.5, 92.0), i, f"{v:.1f}%", va="center", fontsize=8)

    plt.tight_layout()
    plt.show()

    return {
        "predicted_gender": pred_gender_str,
        "gender_sigmoid_prob_male": round(p_male, 4),
        "gender_probabilities": {"Female (0)": round(p_female, 4), "Male (1)": round(p_male, 4)},
        "predicted_age_group": pred_age_group,
        "age_group_probabilities": {AGE_GROUPS_9[i]: round(float(probs_9[i]), 4) for i in range(9)},
    }

IMAGE_URL = ""
LOCAL_IMAGE_PATH = ""
UPLOAD_FROM_COMPUTER = False

result = predict_single_image(
    url=IMAGE_URL,
    local_path=LOCAL_IMAGE_PATH,
    upload_on_colab=UPLOAD_FROM_COMPUTER,
    auto_crop_face=True,
    use_tta=True,
)
print(json.dumps(result, indent=2, ensure_ascii=False))